# 3. Organoid–Cell Relationship

## Purpose
This notebook assigns each single cell and nucleocentric object to its parent organoid,
then computes spatial relationship features (Euclidean distance, Mahalanobis distance,
and shell classification) for each cell relative to its parent organoid centroid.

This is **step 3 of Stage 4 (image-based profiling)**. It runs once per well-FOV and
is typically submitted as a child job via the SLURM scheduler.

## Inputs
Three parquet files from `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/`:
- `sc_profiles_{well_fov}.parquet` — merged Nuclei + Cell + Cytoplasm features
- `organoid_profiles_{well_fov}.parquet` — organoid features
- `nucleocentric_profiles_{well_fov}.parquet` — nucleocentric features

## Outputs
Three enriched parquet files written to `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`:

| File | Added columns |
|---|---|
| `sc_profiles_{well_fov}_related.parquet` | `ParentOrganoid`, shell/distance features |
| `organoid_profiles_{well_fov}_related.parquet` | `OrganoidSingleCellCount` |
| `nucleocentric_profiles_{well_fov}_related.parquet` | `ParentOrganoid` |

## Notes
- Parent organoid assignment uses bbox containment: a cell is assigned to the first
  organoid whose bounding box contains the cell's nuclear centroid.
- Spatial features are computed using Mahalanobis distance with a regularized covariance
  matrix (applied automatically when cell count is low).
- Shell classification divides cells into 4 concentric shells from organoid centroid
  outward, requiring at least 3 cells per shell.

In [1]:
import os
import pathlib

import numpy as np
import pandas as pd
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.neighbors_utils import (
    classify_cells_into_shells,
    euclidean_distance_from_centroid,
    mahalanobis_distance_from_centroid,
)
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm import tqdm

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C10-1"
    image_based_profiles_subparent_name = "image_based_profiles"

### Pathing

In [3]:
# input paths
sc_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve(strict=True)
nucleocentric_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve(strict=True)
# output paths
sc_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_related.parquet"
).resolve()
organoid_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_related.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/nucleocentric_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
sc_profile_df = pd.read_parquet(sc_profile_path)
nucleocentric_df = pd.read_parquet(nucleocentric_profile_path)
organoid_profile_df = pd.read_parquet(organoid_profile_path)
print(f"Single-cell profile shape: {sc_profile_df.shape}")
print(f"Nucleocentric profile shape: {nucleocentric_df.shape}")
print(f"Organoid profile shape: {organoid_profile_df.shape}")

Single-cell profile shape: (13, 11883)
Nucleocentric profile shape: (13, 3074)
Organoid profile shape: (1, 3961)


In [5]:
x_y_z_sc_colnames = [
    x
    for x in sc_profile_df.columns
    if "area" in x.lower() and "center" in x.lower() and "nuclei" in x.lower()
]
x_y_z_sc_colnames

['Nuclei_NoChannel_AreaSizeShape_CenterX',
 'Nuclei_NoChannel_AreaSizeShape_CenterY',
 'Nuclei_NoChannel_AreaSizeShape_CenterZ']

In [6]:
organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]
organoid_bbox_colnames = sorted(organoid_bbox_colnames)

# When sorted alphabetically, the bbox column names fall in this order:
#   [0] = *MaxX, [1] = *MaxY, [2] = *MaxZ, [3] = *MinX, [4] = *MinY, [5] = *MinZ
# This ordering is assumed in the bbox tuple construction below.

In [7]:
# Initialize ParentOrganoid to -1 (sentinel for unassigned cells).
sc_profile_df["ParentOrganoid"] = -1

# Extract single-cell centroids as numpy array for faster access
sc_centroids = sc_profile_df[x_y_z_sc_colnames].values  # (N_cells, 3) array

# reshape the centroids to be in z,y,x order for easier comparison with bbox
sc_centroids = sc_centroids[:, [2, 1, 0]]

# Loop through organoids with progress bar
for organoid_index, organoid_row in tqdm(
    organoid_profile_df.iterrows(),
    total=len(organoid_profile_df),
    desc="Assigning cells to organoids",
):
    # Build the bbox tuple using the sorted column order documented in the cell above:
    # sorted alphabetically gives [MaxX, MaxY, MaxZ, MinX, MinY, MinZ]
    # so indices [5]=MinZ, [4]=MinY, [3]=MinX, [2]=MaxZ, [1]=MaxY, [0]=MaxX.
    organoid_bbox = (
        organoid_row[organoid_bbox_colnames[5]],  # z_min
        organoid_row[organoid_bbox_colnames[4]],  # y_min
        organoid_row[organoid_bbox_colnames[3]],  # x_min
        organoid_row[organoid_bbox_colnames[2]],  # z_max
        organoid_row[organoid_bbox_colnames[1]],  # y_max
        organoid_row[organoid_bbox_colnames[0]],  # x_max
    )

    z_min, y_min, x_min, z_max, y_max, x_max = organoid_bbox

    # Vectorized bbox check - much faster!
    mask = (
        (sc_centroids[:, 0] >= z_min)  # z
        & (sc_centroids[:, 0] <= z_max)  # z
        & (sc_centroids[:, 1] >= y_min)
        & (sc_centroids[:, 1] <= y_max)
        & (sc_centroids[:, 2] >= x_min)
        & (sc_centroids[:, 2] <= x_max)
    )

    # First-match-wins: if organoid bboxes overlap, a cell is assigned to the first
    # organoid whose bbox contains it and is never reassigned to a later one.
    # Both masks are NumPy arrays (positional) to avoid pandas index misalignment.
    unassigned_mask = sc_profile_df["ParentOrganoid"].values == -1
    final_mask = mask & unassigned_mask

    # Assign parent organoid to matching cells
    sc_profile_df.loc[sc_profile_df.index[final_mask], "ParentOrganoid"] = organoid_row[
        "object_id"
    ]

print(f"Assigned {(sc_profile_df['ParentOrganoid'] != -1).sum()} cells to organoids")
print(f"Unassigned cells: {(sc_profile_df['ParentOrganoid'] == -1).sum()}")

Assigning cells to organoids: 100%|██████████| 1/1 [00:00<00:00, 312.87it/s]

Assigned 11 cells to organoids
Unassigned cells: 2


### Add single-cell counts for each organoid

In [8]:
organoid_sc_counts = (
    sc_profile_df["ParentOrganoid"]
    .value_counts()
    .to_frame(name="OrganoidSingleCellCount")
    .reset_index()
)
# merge the organoid profile with the single-cell counts
organoid_profile_df = pd.merge(
    organoid_profile_df,
    organoid_sc_counts,
    left_on="object_id",
    right_on="ParentOrganoid",
    how="left",
).drop(columns=["ParentOrganoid"])
sc_count = organoid_profile_df.pop("OrganoidSingleCellCount")
organoid_profile_df.insert(2, "OrganoidSingleCellCount", sc_count)

### Empty dataframe fallbacks

If either the organoid or SC profile is empty for this well-FOV, a placeholder row
is inserted so that downstream merges always find consistent columns.

In [9]:
# replace NaN with 0 for organoids that have no assigned cells
organoid_profile_df["OrganoidSingleCellCount"] = (
    organoid_profile_df["OrganoidSingleCellCount"].fillna(0).astype(int)
)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C10-1,11,8323552.0,695.398726,908.387099,21.825544,11686500.0,449,979,...,837.461485,851.838893,838.561914,847.985456,835.907707,836.400088,835.762355,848.355497,833.307533,833.548822


In [10]:
if organoid_profile_df.empty:
    # add a row with 0 values
    organoid_profile_df.loc[len(organoid_profile_df)] = [0] * len(
        organoid_profile_df.columns
    )
    organoid_profile_df["image_set"] = well_fov

In [11]:
print(f"Single-cell profile shape: {sc_profile_df.shape}")

Single-cell profile shape: (13, 11884)


In [12]:
if sc_profile_df.empty:
    # add a row with Na values
    sc_profile_df.loc[len(sc_profile_df)] = [None] * len(sc_profile_df.columns)
    sc_profile_df["image_set"] = well_fov

In [13]:
# Propagate ParentOrganoid to nucleocentric profiles.
# Nucleocentric objects share object_id with their parent nucleus, so joining
# on object_id + image_set carries the organoid assignment through.
nucleocentric_df = pd.merge(
    nucleocentric_df,
    sc_profile_df[["object_id", "image_set", "ParentOrganoid"]],
    on=["object_id", "image_set"],
    how="left",
)

## Get single cell and organoid relationships and spatial distributions

In [14]:
x_y_z_organoid_centroid_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and "center" in x.lower()
]
x_y_z_organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]

In [15]:
results = []

# get the organoid id and the single-cells for each

organoid_ids = organoid_profile_df["object_id"]

# organoid_id = organoid_ids[0]

for organoid_id in organoid_ids:
    organoid_centroid = (
        organoid_profile_df.loc[
            organoid_profile_df["object_id"] == organoid_id,
            x_y_z_organoid_centroid_colnames,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
        .to_numpy(dtype=float)
    )
    organoid_bbox = organoid_profile_df.loc[
        organoid_profile_df["object_id"] == organoid_id, x_y_z_organoid_bbox_colnames
    ].values[0]
    single_cells_in_organoid = sc_profile_df[
        sc_profile_df["ParentOrganoid"] == organoid_id
    ]
    if single_cells_in_organoid.empty:
        print(f"No single cells assigned to organoid {organoid_id}")
        continue

    single_cells_centroids = (
        single_cells_in_organoid[x_y_z_sc_colnames]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_rows = ~pd.isna(single_cells_centroids).any(axis=1)
    single_cells_centroids = single_cells_centroids[valid_rows]
    single_cells_in_organoid = single_cells_in_organoid.loc[valid_rows]

    if single_cells_centroids.shape[0] == 0:
        continue

    # convert to a dict with the key being the object_id
    # rename the centroids to z,y.x
    # x_y_z_sc_colnames is alphabetically sorted: [CenterX, CenterY, CenterZ]
    # so index [0]=X, [1]=Y, [2]=Z. The dict remaps them to named z/y/x keys.
    single_cells_centroids_dict = {
        "object_id": single_cells_in_organoid["object_id"].to_numpy(),
        "z": single_cells_in_organoid[x_y_z_sc_colnames[2]].to_numpy(dtype=float),
        "y": single_cells_in_organoid[x_y_z_sc_colnames[1]].to_numpy(dtype=float),
        "x": single_cells_in_organoid[x_y_z_sc_colnames[0]].to_numpy(dtype=float),
    }

    euclidean_distance = euclidean_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    mahalanobis_distance = mahalanobis_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    shell_classification, centroid = classify_cells_into_shells(
        coords=single_cells_centroids_dict,
        n_shells=4,
        method="mahalanobis",
        min_cells_per_shell=3,
        centroid=organoid_centroid,
    )

    shell_classification_df = pd.DataFrame(shell_classification)
    shell_classification_df["ParentOrganoid"] = organoid_id
    results.append(shell_classification_df)

           Reducing to 3 shells for statistical reliability


In [16]:
# Concatenate per-organoid shell results and rename columns to standard feature name format.
# Added columns (under Nuclei_NoChannel_Neighbors_*):
#   ShellAssignments            — shell index (1=innermost, N=outermost) for each cell
#   DistancesFromCenter         — Mahalanobis distance from organoid centroid
#   DistancesFromExterior       — distance from the outermost shell boundary
#   NormalizedDistancesFromCenter — DistancesFromCenter normalized to [0, 1]
#   ShellsUsed                  — total number of shells actually assigned (may be < 4
#                                 if too few cells to fill all shells)
if results:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

else:
    df = pd.DataFrame(columns=["object_id", "ParentOrganoid"])

# rename the columns

df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment="Nuclei",
            feature_type="Neighbors",
            channel="NoChannel",
            measurement=col,
        )
        for col in df.columns
        if col not in ["object_id", "ParentOrganoid"]
    },
    inplace=True,
)

In [17]:
# concat the shell classification with the single cell profile df to get the full single cell profile with the shell classification and the parent organoid id
sc_profile_with_shells_df = pd.merge(
    sc_profile_df,
    df,
    left_on=["object_id", "ParentOrganoid"],
    right_on=["object_id", "ParentOrganoid"],
    how="left",
)

### Save the profiles

In [18]:
organoid_profile_df.to_parquet(organoid_profile_output_path, index=False)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C10-1,11,8323552.0,695.398726,908.387099,21.825544,11686500.0,449,979,...,837.461485,851.838893,838.561914,847.985456,835.907707,836.400088,835.762355,848.355497,833.307533,833.548822


In [19]:
sc_profile_with_shells_df.to_parquet(sc_profile_output_path, index=False)
sc_profile_with_shells_df.head()

,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_DNA_Texture_Variance-3-09-256,Cytoplasm_DNA_Texture_Variance-3-10-256,Cytoplasm_DNA_Texture_Variance-3-11-256,Cytoplasm_DNA_Texture_Variance-3-12-256,ParentOrganoid,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellsUsed
0,1,C10-1,4077.0,248.521707,742.226637,1.000000,5244.0,230,268,720,...,7.748604e-304,7.748604e-304,7.748604e-304,0.0,-1,NaN,NaN,NaN,NaN,NaN
1,2,C10-1,118749.0,605.367405,796.971579,8.613917,259920.0,522,674,736,...,7.748604e-304,7.748604e-304,7.748604e-304,0.0,1,1.0,143.852716,130.890272,0.523590,3.0
2,3,C10-1,25340.0,578.229242,787.458761,17.628414,75710.0,522,656,736,...,7.748604e-304,7.748604e-304,7.748604e-304,0.0,1,1.0,168.433865,106.309123,0.613060,3.0
3,4,C10-1,106061.0,1302.791035,1094.818746,6.935132,164268.0,1246,1363,1044,...,7.748604e-304,7.748604e-304,7.748604e-304,0.0,-1,NaN,NaN,NaN,NaN,NaN
4,5,C10-1,126753.0,569.161266,925.857045,9.330722,228105.0,516,627,860,...,7.748604e-304,7.748604e-304,7.748604e-304,0.0,1,1.0,128.051614,146.691374,0.466078,3.0


In [20]:
nucleocentric_df.to_parquet(nucleocentric_profile_output_path, index=False)
nucleocentric_df.head()

,object_id,image_set,Nucleocentric_Mito_CHAMMI75_Feature0,Nucleocentric_Mito_CHAMMI75_Feature1,Nucleocentric_Mito_CHAMMI75_Feature10,Nucleocentric_Mito_CHAMMI75_Feature100,Nucleocentric_Mito_CHAMMI75_Feature101,Nucleocentric_Mito_CHAMMI75_Feature102,Nucleocentric_Mito_CHAMMI75_Feature103,Nucleocentric_Mito_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99,ParentOrganoid
0,1,C10-1,2.554311,-2.499211,6.270025,1.855958,2.479886,-2.476496,-2.470967,0.746163,...,-0.125312,0.050814,-0.010892,0.035865,-0.024403,0.106305,0.254916,0.266566,0.139579,-1
1,2,C10-1,7.105213,0.659223,1.042990,-3.127599,1.111077,1.992849,0.941766,1.696154,...,-0.055770,0.022011,-0.010457,0.021449,0.003285,-0.054165,0.223486,0.396483,0.231863,1
2,3,C10-1,4.893235,0.385423,2.861571,-1.543698,2.498050,2.142777,-0.935785,-0.771587,...,-0.049796,0.002883,-0.010533,0.031659,-0.033706,-0.029345,0.212940,0.351991,0.266070,1
3,4,C10-1,0.144605,-4.757784,7.886053,1.660694,1.476611,-5.204203,0.042312,4.300991,...,-0.090895,0.086051,-0.010847,0.027819,0.016349,-0.029114,0.257279,0.364449,0.183671,-1
4,5,C10-1,0.903223,-0.710765,2.194714,1.150269,0.404250,0.147449,1.135373,-0.689170,...,-0.086117,0.021421,-0.010490,-0.000571,-0.032310,-0.092488,0.232491,0.379522,0.152363,1
